In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers==4.52.4 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 70.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 86.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled hug

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/tmp/ipykernel_57/3926598702.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [6]:
pdf_path = "/kaggle/input/datasets/basemhesham/basemhesham/Basem Hesham CV.pdf"
loader = PyPDFLoader(pdf_path)
resume_text = loader.load()
print(resume_text)

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-10-01T18:11:55+03:00', 'author': 'Un-named', 'moddate': '2026-10-01T18:11:55+03:00', 'source': '/kaggle/input/datasets/basemhesham/basemhesham/Basem Hesham CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='BASEM HESHAM REZK \nJunior AI & Machine Learning Engineer | Computer and Communication Engineering \nbasemhesham200318@gmail.com   |   +201223868241   |   LinkedIn   |   GitHub   |   Alexandria, Egypt    \nOBJECTIVE \nComputer and Communication Engineering graduate specializing in Artificial Intelligence with hands-on experience in Machine \nLearning, Deep Learning, NLP, and Computer Vision. Experienced in developing AI applications using Python, TensorFlow, \nPyTorch, Hugging Face, and Scikit-learn through internships and academic projects. Passionate about building intelligent \nsystems and solving real-world problems using modern AI technologies.  

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
import torch
import re

In [8]:
response_schemas = [
    ResponseSchema(
        name="full_name",
        description="The candidate's full name as a string"
    ),
    ResponseSchema(
        name="email",
        description="The candidate's email address as a string"
    ),
    ResponseSchema(
        name="education",
        description="A list of education entries. Each entry contains degree, institution, and year."
    ),
    ResponseSchema(
        name="skills",
        description="A list of the candidate's skills as strings"
    ),
    ResponseSchema(
        name="experience",
        description="A list of work experience entries. Each entry contains role, company, and years."
    )
]

output_parser = StructuredOutputParser.from_response_schemas(
    response_schemas
)
format_instructions = output_parser.get_format_instructions()

print(format_instructions)

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name as a string
	"email": string  // The candidate's email address as a string
	"education": string  // A list of education entries. Each entry contains degree, institution, and year.
	"skills": string  // A list of the candidate's skills as strings
	"experience": string  // A list of work experience entries. Each entry contains role, company, and years.
}
```


In [9]:
print(type(output_parser))
print(type(format_instructions))

<class 'langchain_classic.output_parsers.structured.StructuredOutputParser'>
<class 'str'>


In [15]:
template ="""
You are an HR Candidate Profile Parser.

Extract the relevant information from the resume text below.

Resume:
{resume_text}

Extract:
- Full name
- Email
- Education
- Skills
- Work experience

Education must contain:
- degree
- institution
- year

Experience must contain:
- role
- company
- years

{format_instructions}
"""

In [16]:
print(template)


You are an HR Candidate Profile Parser.

Extract the relevant information from the resume text below.

Resume:
{resume_text}

Extract:
- Full name
- Email
- Education
- Skills
- Work experience

Education must contain:
- degree
- institution
- year

Experience must contain:
- role
- company
- years

{format_instructions}



In [17]:
formatted_prompt= PromptTemplate(template=template,input_variables=["resume_text","format_instructions"]).format(resume_text=resume_text,format_instructions=format_instructions)
print(formatted_prompt)


You are an HR Candidate Profile Parser.

Extract the relevant information from the resume text below.

Resume:
[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-10-01T18:11:55+03:00', 'author': 'Un-named', 'moddate': '2026-10-01T18:11:55+03:00', 'source': '/kaggle/input/datasets/basemhesham/basemhesham/Basem Hesham CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='BASEM HESHAM REZK \nJunior AI & Machine Learning Engineer | Computer and Communication Engineering \nbasemhesham200318@gmail.com   |   +201223868241   |   LinkedIn   |   GitHub   |   Alexandria, Egypt    \nOBJECTIVE \nComputer and Communication Engineering graduate specializing in Artificial Intelligence with hands-on experience in Machine \nLearning, Deep Learning, NLP, and Computer Vision. Experienced in developing AI applications using Python, TensorFlow, \nPyTorch, Hugging Face, and Scikit-learn through internships and academic projects.

In [18]:
response = generate_text(formatted_prompt, max_length=3000)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [23]:
print(response)


You are an HR Candidate Profile Parser.

Extract the relevant information from the resume text below.

Resume:
[Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-10-01T18:11:55+03:00', 'author': 'Un-named', 'moddate': '2026-10-01T18:11:55+03:00', 'source': '/kaggle/input/datasets/basemhesham/basemhesham/Basem Hesham CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='BASEM HESHAM REZK \nJunior AI & Machine Learning Engineer | Computer and Communication Engineering \nbasemhesham200318@gmail.com   |   +201223868241   |   LinkedIn   |   GitHub   |   Alexandria, Egypt    \nOBJECTIVE \nComputer and Communication Engineering graduate specializing in Artificial Intelligence with hands-on experience in Machine \nLearning, Deep Learning, NLP, and Computer Vision. Experienced in developing AI applications using Python, TensorFlow, \nPyTorch, Hugging Face, and Scikit-learn through internships and academic projects.

In [24]:
def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"
response=extract_json_block(response)
parsed_result = output_parser.parse(response)

print(parsed_result)

{'full_name': 'BASEM HESHAM REZK', 'email': 'basemhesham200318@gmail.com', 'education': [{'degree': 'Bachelor of computer and communication Engineering (Minor Ai)', 'institution': 'Alexandria University, Egypt', 'year': '2026'}], 'skills': 'Python, C, Java, SQL, Deep Learning, NLP, Computer Vision, Transformers, CNN, RNN, Transfer Learning, Model Evaluation , Optimization, TensorFlow, Keras, PyTorch, Scikit-learn, Hugging Face, OpenCV, Pandas, NumPy, Git, GitHub, Jupyter, Google Colab, Linux, MySQL, VS Code, Data Structures & Algorithms, Object-Oriented Programming (OOP), Database Systems, Operating Systems, Computer Organization, Computer Networks, Cybersecurity Fundamentals, System Programming, Software Engineering, Data Analysis, Data Modeling, Machine Learning Algorithms, SQL Databases, System Design, Secure Application Development, Data Processing, Software Optimization, Fortinet Security Fabric, Wireshark, TCP/IP, Routing & Switching, VLANs, Laravel, RESTful APIs, MVC Architectur

---
# Part 2 Final Project - AI Career Coach (RAG + Chains + Output Parsers)

Everything above (Mistral-7B, PDF loader, `StructuredOutputParser`) is reused. The new code turns it into a career coach that helps a user pick the most suitable career path.

**Inputs:** CV, skills, experience, target role  
**RAG knowledge base:** job descriptions, learning roadmaps, industry requirements  
**Chains:** analyse CV -> extract skills -> compare with job requirements -> build plan  
**Output:** current skills, missing skills, recommended roadmap, suggested jobs, improvement plan

```
 CV + skills + experience + target role
              |
   Chain 1: Analyse CV ........... profile (parser 1)
   Chain 2: Extract skills ....... skills  (parser 2)
              |
   RAG  <- job descriptions + industry requirements
   Chain 3: Compare with job requirements ... gap (parser 3)
              |
   RAG  <- learning roadmaps
   Chain 4: Roadmap + jobs + improvement plan ... plan (parser 4)
              |
   Final report (dict)  ->  Streamlit app
```

In [25]:
!pip install -q streamlit pyngrok bitsandbytes accelerate langchain-classic langchain-text-splitters

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 45.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 96.4 MB/s eta 0:00:00:00:010:01


## Step 1 - Configuration
Paths, model names and the three document types of the knowledge base.

In [28]:
import os
import re
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
BASE_DIR = Path(os.environ.get(
    "COACH_BASE_DIR", "/kaggle/working" if os.path.exists("/kaggle/working") else "."))
KB_DIR = BASE_DIR / "knowledge_base"          # job descriptions / roadmaps / industry docs
INDEX_DIR = BASE_DIR / "faiss_career_index"   # saved FAISS index
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_NAME = os.environ.get("COACH_MODEL", "mistralai/Mistral-7B-Instruct-v0.2")
MAX_CV_CHARS = 6000                           # keep prompts short enough for a 7B model

# folder name -> metadata value used for filtered retrieval
DOC_TYPES = {
    "job_descriptions": "job_description",
    "learning_roadmaps": "learning_roadmap",
    "industry_requirements": "industry_requirement",
}


## Step 2 - Build the RAG knowledge base
The knowledge base is just files in three folders, so you can grow it later:

| Folder | Content |
|---|---|
| `knowledge_base/job_descriptions/` | responsibilities and required skills per role |
| `knowledge_base/learning_roadmaps/` | phases, durations and projects per role |
| `knowledge_base/industry_requirements/` | certificates, portfolio and hiring expectations |

The cell below writes a **starter set for 7 roles**. To add your own roles or real job postings, drop `.md`, `.txt` or `.pdf` files into these folders (start the file with a line like `Role: Cloud Engineer`) and run the indexing cell again with `rebuild=True`.

In [29]:
"""Seed knowledge base: {folder: {file_name: text}}.  Each file starts with a 'Role:' line."""

ROLES = {
    "data_analyst": "Data Analyst",
    "data_scientist": "Data Scientist",
    "ml_engineer": "Machine Learning Engineer",
    "ai_llm_engineer": "AI / LLM Engineer",
    "backend_developer": "Backend Developer",
    "frontend_developer": "Frontend Developer",
    "devops_engineer": "DevOps Engineer",
}

JOB_DESCRIPTIONS = {
"data_analyst": """Role: Data Analyst
Doc type: Job description

Summary: Turns raw business data into reports, dashboards and recommendations that help teams make decisions.
Responsibilities: write SQL queries to extract and clean data; build dashboards and recurring reports; analyse trends, run A/B test readouts; present findings to non-technical stakeholders; define and track KPIs.
Required skills: SQL (joins, window functions, aggregations), Excel / Google Sheets, a BI tool (Power BI, Tableau or Looker), descriptive statistics, data cleaning, data visualisation, clear communication.
Nice to have: Python (pandas, matplotlib), basic A/B testing, basic ETL, domain knowledge (finance, e-commerce, marketing).
Typical titles: Data Analyst, Business Intelligence Analyst, Reporting Analyst, Product Analyst.""",

"data_scientist": """Role: Data Scientist
Doc type: Job description

Summary: Uses statistics and machine learning to answer business questions and build predictive models.
Responsibilities: explore and clean datasets; engineer features; train, validate and compare ML models; design experiments and A/B tests; communicate results and model limitations; work with engineers to hand models over to production.
Required skills: Python (pandas, NumPy, scikit-learn), SQL, statistics and probability, hypothesis testing, supervised and unsupervised learning, model evaluation (cross-validation, ROC-AUC, precision/recall), data visualisation, storytelling.
Nice to have: deep learning (PyTorch or TensorFlow), NLP, time-series forecasting, Spark, cloud basics (AWS, GCP, Azure), Git.
Typical titles: Data Scientist, Applied Scientist, Junior Data Scientist, Decision Scientist.""",

"ml_engineer": """Role: Machine Learning Engineer
Doc type: Job description

Summary: Builds, deploys and maintains machine learning systems in production.
Responsibilities: turn prototypes into reliable services; build training and inference pipelines; deploy models behind APIs; monitor model drift and latency; optimise performance and cost; collaborate with data scientists and backend engineers.
Required skills: strong Python, PyTorch or TensorFlow, scikit-learn, software engineering practices (testing, Git, code review), Docker, REST APIs (FastAPI or Flask), SQL, MLOps basics (experiment tracking, model registry, CI/CD).
Nice to have: Kubernetes, cloud ML services (SageMaker, Vertex AI, Azure ML), Spark, feature stores, model optimisation (quantisation, ONNX), Airflow.
Typical titles: Machine Learning Engineer, MLOps Engineer, Applied ML Engineer.""",

"ai_llm_engineer": """Role: AI / LLM Engineer
Doc type: Job description

Summary: Builds applications on top of large language models, such as chatbots, assistants, RAG systems and agents.
Responsibilities: design prompts and chains; build retrieval-augmented generation (RAG) pipelines with vector databases; parse and validate structured LLM output; evaluate answer quality and hallucination rate; fine-tune or adapt open models; deploy LLM apps and keep latency and cost under control.
Required skills: Python, LangChain or LlamaIndex, Hugging Face Transformers, embeddings and vector stores (FAISS, Chroma, Pinecone), prompt engineering, output parsing / JSON schemas, REST APIs, Git.
Nice to have: fine-tuning (LoRA / QLoRA), evaluation frameworks, agents and tool calling, FastAPI, Docker, Streamlit or Gradio for demos, cloud GPUs, quantisation.
Typical titles: LLM Engineer, Generative AI Engineer, AI Engineer, NLP Engineer.""",

"backend_developer": """Role: Backend Developer
Doc type: Job description

Summary: Designs and builds the server-side logic, APIs and databases that power applications.
Responsibilities: design and implement REST or GraphQL APIs; model data and write database queries; handle authentication and authorisation; write automated tests; monitor, debug and optimise services; collaborate with frontend and DevOps teams.
Required skills: one backend language and framework (Python with Django or FastAPI, Node.js with Express, Java with Spring, or C# with .NET), SQL and relational design (PostgreSQL or MySQL), REST API design, Git, authentication (JWT, OAuth), unit and integration testing.
Nice to have: Docker, Redis caching, message queues (RabbitMQ, Kafka), NoSQL (MongoDB), cloud basics, CI/CD, system design basics.
Typical titles: Backend Developer, Software Engineer (Backend), API Developer.""",

"frontend_developer": """Role: Frontend Developer
Doc type: Job description

Summary: Builds the user interfaces that people see and interact with in the browser.
Responsibilities: implement responsive, accessible interfaces; connect UIs to APIs; manage application state; write component and end-to-end tests; optimise page performance; work with designers and backend engineers.
Required skills: HTML, CSS, modern JavaScript (ES6+), a framework (React, Vue or Angular), responsive design, Git, consuming REST APIs, browser dev tools, basic accessibility.
Nice to have: TypeScript, Next.js, state management (Redux, Zustand), Tailwind CSS, testing (Jest, Cypress, Playwright), web performance, UI/UX sense.
Typical titles: Frontend Developer, React Developer, UI Engineer.""",

"devops_engineer": """Role: DevOps Engineer
Doc type: Job description

Summary: Automates building, testing, deploying and operating software reliably.
Responsibilities: build and maintain CI/CD pipelines; manage cloud infrastructure with code; containerise and orchestrate applications; set up monitoring, logging and alerting; improve reliability, security and deployment speed; respond to incidents.
Required skills: Linux and shell scripting, Git, CI/CD (GitHub Actions, GitLab CI or Jenkins), Docker, Kubernetes, one cloud provider (AWS, Azure or GCP), Infrastructure as Code (Terraform), networking basics, monitoring (Prometheus, Grafana).
Nice to have: Ansible, Helm, scripting in Python or Go, security practices (secrets management, scanning), cost optimisation.
Typical titles: DevOps Engineer, Site Reliability Engineer, Platform Engineer, Cloud Engineer.""",
}

LEARNING_ROADMAPS = {
"data_analyst": """Role: Data Analyst
Doc type: Learning roadmap

Phase 1 (4 weeks) - Foundations: Excel / Sheets (pivot tables, lookups), descriptive statistics, data types and cleaning. Project: clean and summarise a messy public CSV.
Phase 2 (6 weeks) - SQL: SELECT, joins, GROUP BY, subqueries, CTEs, window functions. Project: answer 20 business questions on a sample sales database.
Phase 3 (4 weeks) - Visualisation and BI: Power BI or Tableau, chart choice, dashboard design. Project: interactive sales dashboard.
Phase 4 (4 weeks) - Python for analysis: pandas, matplotlib, EDA workflow. Project: end-to-end EDA notebook with written insights.
Phase 5 (3 weeks) - Business skills: KPIs, A/B test basics, storytelling. Project: 5-slide executive summary and a portfolio on GitHub.""",

"data_scientist": """Role: Data Scientist
Doc type: Learning roadmap

Phase 1 (4 weeks) - Python and data handling: pandas, NumPy, matplotlib, SQL basics. Project: EDA on a public dataset.
Phase 2 (5 weeks) - Statistics: distributions, hypothesis tests, confidence intervals, correlation vs causation. Project: A/B test analysis.
Phase 3 (8 weeks) - Machine learning: scikit-learn, regression, trees, ensembles, clustering, feature engineering, cross-validation, metrics. Project: Kaggle-style prediction competition with a written report.
Phase 4 (6 weeks) - Deep learning and specialisation: PyTorch basics, then NLP or time series or computer vision. Project: one specialised model with error analysis.
Phase 5 (4 weeks) - Delivery: Git, reproducible pipelines, simple model deployment with Streamlit or FastAPI, communication. Project: deployed demo plus a case-study blog post.""",

"ml_engineer": """Role: Machine Learning Engineer
Doc type: Learning roadmap

Phase 1 (4 weeks) - Engineering foundations: strong Python, OOP, Git, testing with pytest, Linux shell. Project: packaged Python library with tests.
Phase 2 (6 weeks) - ML and deep learning: scikit-learn, PyTorch, training loops, evaluation. Project: image or text classifier.
Phase 3 (4 weeks) - Serving: FastAPI, Docker, REST design, batch vs online inference. Project: containerised model API.
Phase 4 (6 weeks) - MLOps: MLflow or Weights & Biases, data versioning (DVC), CI/CD with GitHub Actions, monitoring and drift detection. Project: automated train-and-deploy pipeline.
Phase 5 (5 weeks) - Scale and cloud: one cloud ML platform, Kubernetes basics, model optimisation (ONNX, quantisation). Project: deploy the pipeline on a cloud service.""",

"ai_llm_engineer": """Role: AI / LLM Engineer
Doc type: Learning roadmap

Phase 1 (4 weeks) - Python and ML basics: Python, APIs, JSON, scikit-learn essentials, neural network fundamentals. Project: small text classifier.
Phase 2 (4 weeks) - Transformers and LLM basics: tokenisation, attention, Hugging Face Transformers, prompting and sampling parameters. Project: local chatbot with an open model.
Phase 3 (5 weeks) - RAG: embeddings, chunking strategies, vector stores (FAISS, Chroma), retrieval evaluation, LangChain chains. Project: question answering over your own PDFs.
Phase 4 (4 weeks) - Structured output and agents: output parsers, JSON schemas, tool calling, simple agents, guardrails. Project: an agent that calls two tools and returns validated JSON.
Phase 5 (5 weeks) - Evaluation and deployment: LLM evaluation, fine-tuning with LoRA / QLoRA, quantisation, FastAPI or Streamlit app, Docker. Project: deployed RAG app with an evaluation report.""",

"backend_developer": """Role: Backend Developer
Doc type: Learning roadmap

Phase 1 (4 weeks) - Language fundamentals: one language (Python or JavaScript or Java), data structures, Git. Project: command-line app.
Phase 2 (5 weeks) - Databases: SQL, PostgreSQL, schema design, indexing, ORMs. Project: normalised schema for an e-commerce system.
Phase 3 (6 weeks) - APIs and frameworks: FastAPI / Django / Express, REST conventions, validation, authentication with JWT. Project: full CRUD API with auth.
Phase 4 (4 weeks) - Quality and delivery: unit and integration tests, Docker, CI with GitHub Actions. Project: containerised API with a test pipeline.
Phase 5 (5 weeks) - Scaling concepts: caching with Redis, queues, basic system design, cloud deployment. Project: deploy the API with caching and a background job.""",

"frontend_developer": """Role: Frontend Developer
Doc type: Learning roadmap

Phase 1 (4 weeks) - Web basics: semantic HTML, CSS (flexbox, grid), responsive design. Project: responsive portfolio page.
Phase 2 (5 weeks) - JavaScript: ES6+, DOM, async/await, fetch. Project: interactive app using a public API.
Phase 3 (6 weeks) - React: components, hooks, routing, forms, state management. Project: multi-page React app.
Phase 4 (4 weeks) - TypeScript and tooling: TypeScript, Vite or Next.js, Tailwind. Project: migrate the React app to TypeScript.
Phase 5 (4 weeks) - Quality: component and end-to-end tests, accessibility audits, performance optimisation, deployment on Vercel or Netlify. Project: polished deployed app with test coverage.""",

"devops_engineer": """Role: DevOps Engineer
Doc type: Learning roadmap

Phase 1 (4 weeks) - Foundations: Linux, shell scripting, networking basics, Git. Project: automate server setup with a shell script.
Phase 2 (4 weeks) - Containers: Docker, Docker Compose, image best practices. Project: containerise a multi-service app.
Phase 3 (4 weeks) - CI/CD: GitHub Actions or GitLab CI, build-test-deploy pipelines. Project: pipeline that tests and deploys on every merge.
Phase 4 (6 weeks) - Cloud and IaC: one cloud (AWS, Azure or GCP), Terraform, IAM and networking. Project: provision a full environment with Terraform.
Phase 5 (6 weeks) - Orchestration and observability: Kubernetes, Helm, Prometheus and Grafana, logging, incident basics. Project: deploy an app to Kubernetes with dashboards and alerts.""",
}

INDUSTRY_REQUIREMENTS = {
"data_analyst": """Role: Data Analyst
Doc type: Industry requirements

Hiring expectations: entry-level employers expect strong SQL and one BI tool; most screening includes a SQL test and a take-home dashboard or case study.
Useful certifications: Microsoft PL-300 (Power BI Data Analyst), Tableau Desktop Specialist, Google Data Analytics Certificate.
Portfolio expectations: 3 projects showing the full flow (question, data, analysis, recommendation), published dashboards, a clear README.
Market trends: self-service BI, growing use of AI-assisted analytics, demand for analysts who understand the business domain.""",

"data_scientist": """Role: Data Scientist
Doc type: Industry requirements

Hiring expectations: employers expect solid statistics, Python and SQL; interviews include ML theory, a coding round, a case study and a take-home project. Many roles prefer a bachelor's or master's in a quantitative field, but strong portfolios count.
Useful certifications: AWS Certified Machine Learning Specialty, Google Professional Data Engineer or ML Engineer (for cloud exposure), TensorFlow Developer.
Portfolio expectations: 2-3 end-to-end projects with clear problem framing, proper validation, error analysis and a deployed demo; Kaggle results are a plus.
Market trends: LLM-assisted workflows, stronger emphasis on experimentation and business impact, closer overlap with ML engineering.""",

"ml_engineer": """Role: Machine Learning Engineer
Doc type: Industry requirements

Hiring expectations: employers expect software engineering quality (tests, code review, Git) as well as ML knowledge; interviews include coding, ML system design and practical deployment questions.
Useful certifications: AWS Certified Machine Learning Specialty, Google Professional Machine Learning Engineer, Azure AI Engineer Associate, Certified Kubernetes Application Developer.
Portfolio expectations: a deployed model service with Docker and CI/CD, monitoring, and documentation; open-source contributions are a plus.
Market trends: MLOps maturity, model monitoring, efficient inference, feature stores, growing demand for LLM deployment skills.""",

"ai_llm_engineer": """Role: AI / LLM Engineer
Doc type: Industry requirements

Hiring expectations: employers want people who ship working LLM applications: RAG, structured output, evaluation, and cost/latency awareness. Interviews focus on prompt design, retrieval quality, hallucination handling and system design for LLM apps.
Useful certifications: cloud AI certificates (Azure AI Engineer, Google Professional ML Engineer, AWS ML Specialty) and short courses on LangChain, Hugging Face and LLM evaluation; none are mandatory.
Portfolio expectations: 2-3 public projects such as a RAG chatbot over real documents, an agent with tools, and a fine-tuned model, each with an evaluation section and a live demo (Streamlit, Gradio or a hosted API).
Market trends: agents and tool calling, evaluation and guardrails, small open models running locally, multimodal inputs, strong demand for engineers who can combine LLMs with classic software engineering.""",

"backend_developer": """Role: Backend Developer
Doc type: Industry requirements

Hiring expectations: employers expect solid data structures, one backend stack, SQL and API design; interviews include coding rounds, a take-home API task and basic system design for mid-level roles.
Useful certifications: AWS Certified Developer Associate, Microsoft Azure Developer Associate, Oracle Java certifications (optional).
Portfolio expectations: a documented API with authentication, tests, Docker and a live deployment; clean Git history.
Market trends: cloud-native services, microservices with pragmatic limits, observability, API security, serverless for some workloads.""",

"frontend_developer": """Role: Frontend Developer
Doc type: Industry requirements

Hiring expectations: employers expect strong JavaScript and React (or an equivalent framework), responsive and accessible UI, and comfort with APIs; interviews include a UI build task and JavaScript fundamentals.
Useful certifications: few are required; Meta Front-End Developer certificate or similar are optional. Deployed projects matter more.
Portfolio expectations: 3 deployed apps with clean code, responsive layouts, accessibility and some tests; a public GitHub profile.
Market trends: TypeScript as default, Next.js and server components, performance and Core Web Vitals, design systems.""",

"devops_engineer": """Role: DevOps Engineer
Doc type: Industry requirements

Hiring expectations: employers expect hands-on cloud, CI/CD, containers and scripting; interviews include Linux troubleshooting, pipeline design and scenario questions about incidents.
Useful certifications: AWS Certified Solutions Architect Associate or SysOps, HashiCorp Terraform Associate, Certified Kubernetes Administrator (CKA), Azure Administrator.
Portfolio expectations: infrastructure-as-code repositories, a CI/CD pipeline with automated tests and deployment, a Kubernetes demo with monitoring.
Market trends: platform engineering, GitOps, security shifted left (DevSecOps), FinOps and cost control, observability.""",
}

SEED_KB = {
    "job_descriptions": JOB_DESCRIPTIONS,
    "learning_roadmaps": LEARNING_ROADMAPS,
    "industry_requirements": INDUSTRY_REQUIREMENTS,
}

# Write the starter files to disk
for folder, files in SEED_KB.items():
    out_dir = KB_DIR / folder
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, text in files.items():
        (out_dir / f"{name}.md").write_text(text, encoding="utf-8")

print("Knowledge-base files written:", sum(len(f) for f in SEED_KB.values()))

Knowledge-base files written: 21


Retrieval helpers. Every chunk carries metadata (`role`, `doc_type`), so we can ask FAISS for *"the learning roadmap of this role"* instead of hoping that plain similarity search finds it.

In [31]:
def load_kb_documents(kb_dir=KB_DIR):
    """Read every .md / .txt / .pdf under knowledge_base/<doc-type folder>/ into LangChain Documents.
    A file may start with a line 'Role: <role name>'; otherwise the file name is used as the role."""
    docs = []
    for folder, doc_type in DOC_TYPES.items():
        for path in sorted((Path(kb_dir) / folder).glob("*")):
            suffix = path.suffix.lower()
            if suffix in {".md", ".txt"}:
                text = path.read_text(encoding="utf-8")
            elif suffix == ".pdf":
                text = "\n".join(p.page_content for p in PyPDFLoader(str(path)).load())
            else:
                continue
            match = re.search(r"^Role:\s*(.+)$", text, re.MULTILINE)
            role = match.group(1).strip() if match else path.stem.replace("_", " ").title()
            docs.append(Document(
                page_content=text,
                metadata={"role": role, "doc_type": doc_type, "source": path.name},
            ))
    return docs


def list_roles(kb_dir=KB_DIR):
    """All roles that have a job description in the knowledge base."""
    return sorted({d.metadata["role"] for d in load_kb_documents(kb_dir)
                   if d.metadata["doc_type"] == "job_description"})


_VECTORSTORE = None


def get_vectorstore(rebuild=False):
    """Load the FAISS index from disk (or build it from the knowledge base)."""
    global _VECTORSTORE
    if _VECTORSTORE is not None and not rebuild:
        return _VECTORSTORE
    embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cpu"})
    if INDEX_DIR.exists() and not rebuild:
        _VECTORSTORE = FAISS.load_local(
            str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
        return _VECTORSTORE
    docs = load_kb_documents()
    if not docs:
        raise FileNotFoundError(
            f"No knowledge-base files found in {KB_DIR}. Run the 'Build the knowledge base' "
            "cells in the notebook first.")
    splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    _VECTORSTORE = FAISS.from_documents(chunks, embeddings)
    _VECTORSTORE.save_local(str(INDEX_DIR))
    return _VECTORSTORE


def retrieve(query, doc_type, role=None, k=4):
    """Semantic search restricted to one document type (and optionally one role)."""
    metadata_filter = {"doc_type": doc_type}
    if role:
        metadata_filter["role"] = role
    return get_vectorstore().similarity_search(query, k=k, filter=metadata_filter, fetch_k=500)


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['role']} | {d.metadata['doc_type']}]\n{d.page_content}"
                       for d in docs)


def resolve_role(text):
    """Map free text ('senior data scientist', 'NLP engineer') to the closest role in the knowledge base."""
    roles = list_roles()
    wanted = text.strip().lower()
    for role in roles:                                   # exact match
        if role.lower() == wanted:
            return role
    for role in roles:                                   # containment match
        if len(wanted) >= 4 and (role.lower() in wanted or wanted in role.lower()):
            return role
    hit = get_vectorstore().similarity_search(           # semantic fallback
        text, k=1, filter={"doc_type": "job_description"}, fetch_k=500)
    return hit[0].metadata["role"]


def best_fit_roles(query, k=3):
    """Roles whose job descriptions are semantically closest to a skills description."""
    docs = get_vectorstore().similarity_search(
        query, k=30, filter={"doc_type": "job_description"}, fetch_k=500)
    roles = []
    for d in docs:
        if d.metadata["role"] not in roles:
            roles.append(d.metadata["role"])
    return roles[:k]


def role_snapshot(role):
    """One-line description of a role, taken from its job description."""
    docs = retrieve(f"{role} summary", "job_description", role=role, k=1)
    if not docs:
        return role
    m = re.search(r"Summary:\s*(.+)", docs[0].page_content)
    return f"{role}: {m.group(1).strip() if m else docs[0].page_content[:200]}"


In [32]:
vectorstore = get_vectorstore(rebuild=True)      # chunk -> embed -> FAISS -> save to disk
print("Chunks indexed:", vectorstore.index.ntotal)
print("Roles:", list_roles())

# Quick retrieval test (no role filter -> pure semantic search inside one document type)
for d in retrieve("how do I become an LLM engineer?", "learning_roadmap", k=2):
    print("\n", d.metadata)
    print(d.page_content[:300])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chunks indexed: 24
Roles: ['AI / LLM Engineer', 'Backend Developer', 'Data Analyst', 'Data Scientist', 'DevOps Engineer', 'Frontend Developer', 'Machine Learning Engineer']

 {'role': 'AI / LLM Engineer', 'doc_type': 'learning_roadmap', 'source': 'ai_llm_engineer.md'}
Role: AI / LLM Engineer
Doc type: Learning roadmap

 {'role': 'Data Scientist', 'doc_type': 'learning_roadmap', 'source': 'data_scientist.md'}
Role: Data Scientist
Doc type: Learning roadmap

Phase 1 (4 weeks) - Python and data handling: pandas, NumPy, matplotlib, SQL basics. Project: EDA on a public dataset.
Phase 2 (5 weeks) - Statistics: distributions, hypothesis tests, confidence intervals, correlation vs causation. Project: A/B test a


## Step 3 - LLM wrapper
The lab's `generate_text` used `max_length` and returned the prompt together with the answer. `llm_generate` uses the model's chat template, decodes **only the new tokens** and uses greedy decoding so JSON output is stable. It reuses the `model` and `tokenizer` you already loaded above.

In [33]:
def load_llm(model_name=MODEL_NAME, use_4bit=False):
    """Load tokenizer + model into the module globals `tokenizer` / `model`.
    (In the notebook the model is already loaded, so this is only used by the Streamlit app.)"""
    global tokenizer, model
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    kwargs = {"device_map": "auto"}
    if use_4bit and torch.cuda.is_available():
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    else:
        kwargs["torch_dtype"] = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    return model, tokenizer


def llm_generate(prompt, max_new_tokens=700):
    """Greedy generation that returns ONLY the new text (the lab's generate_text echoed the prompt too).
    Greedy decoding keeps JSON output stable and repeatable."""
    import torch
    if model is None:
        load_llm()
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [34]:
print(llm_generate("In one sentence, what does a Machine Learning Engineer do?", max_new_tokens=60))

A Machine Learning Engineer designs, builds, and implements machine learning systems, utilizing algorithms, data modeling, and programming to develop predictive models and improve business processes.


## Step 4 - Output parsers
Same `StructuredOutputParser` idea as the lab, one parser per chain. Descriptions tell the model the exact shape of each field (e.g. *list of objects with keys skill, priority, reason*).

In [35]:
def make_parser(fields):
    """Build a StructuredOutputParser from (name, description) pairs."""
    return StructuredOutputParser.from_response_schemas(
        [ResponseSchema(name=n, description=d) for n, d in fields])


# Parser 1 - CV analysis (extends the lab's original schema)
profile_parser = make_parser([
    ("full_name", "The candidate's full name as a string, or \"Unknown\""),
    ("email", "The candidate's email address as a string, or \"Unknown\""),
    ("education", "List of objects, each with the keys degree, institution, year"),
    ("experience", "List of objects, each with the keys role, company, years. Include internships and training programs"),
    ("summary", "Two-sentence professional summary of the candidate, as a string"),
    ("experience_level", "Exactly one of: Student, Fresher, Junior, Mid-level, Senior"),
])

# Parser 2 - skill extraction
skills_parser = make_parser([
    ("technical_skills", "List of strings: programming languages and technical concepts (e.g. Python, SQL, machine learning, NLP)"),
    ("tools_and_technologies", "List of strings: frameworks, libraries, platforms and tools (e.g. PyTorch, Docker, LangChain)"),
    ("soft_skills", "List of strings: soft skills such as communication or teamwork"),
    ("domains", "List of strings: fields the candidate has worked or studied in"),
    ("notable_projects", "List of at most 5 short strings, one per project"),
])

# Parser 3 - CV vs. job requirements
gap_parser = make_parser([
    ("matched_skills", "List of strings: candidate skills that the target role requires"),
    ("missing_skills", "List of objects, each with the keys skill, priority (High, Medium or Low) and reason (one short sentence)"),
    ("match_score", "Integer from 0 to 100: how ready the candidate is for the target role today"),
    ("readiness_summary", "Two or three honest sentences explaining the score"),
])

# Parser 4 - roadmap, jobs and improvement plan
plan_parser = make_parser([
    ("recommended_roadmap", "List of 4 to 6 phases, each an object with the keys phase (short name), duration (e.g. '4 weeks'), topics (list of strings) and project (one practical project)"),
    ("suggested_jobs", "List of exactly 3 objects, each with the keys title, why_it_fits and next_step"),
    ("improvement_plan", "List of 3 objects, each with the keys timeframe (e.g. 'Next 30 days') and actions (list of 2 to 4 concrete strings)"),
])


In [36]:
print(gap_parser.get_format_instructions())

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"matched_skills": string  // List of strings: candidate skills that the target role requires
	"missing_skills": string  // List of objects, each with the keys skill, priority (High, Medium or Low) and reason (one short sentence)
	"match_score": string  // Integer from 0 to 100: how ready the candidate is for the target role today
	"readiness_summary": string  // Two or three honest sentences explaining the score
}
```


## Step 5 - Normalisers
A 7B model sometimes returns a string where a list was requested, or `"62%"` instead of `62`. These small functions make the final report always have the expected types, which also keeps the Streamlit UI from crashing.

In [37]:
_EMPTY = {"", "none", "n/a", "unknown", "not provided", "null"}


def _clean(value):
    return re.sub(r"\s+", " ", str(value)).strip()


def _flatten(value):
    if value is None:
        return []
    if isinstance(value, dict):
        name = value.get("skill") or value.get("name") or value.get("title")
        return [name] if name else [x for v in value.values() for x in _flatten(v)]
    if isinstance(value, (list, tuple, set)):
        return [x for v in value for x in _flatten(v)]
    return [value]


def to_str_list(value):
    """Any LLM answer -> clean, de-duplicated list of strings."""
    if isinstance(value, str):
        value = re.split(r"[\n,;\u2022]+", value)
    out, seen = [], set()
    for item in _flatten(value):
        text = _clean(item).strip("-*\u2022 ")
        if text.lower() not in _EMPTY and text.lower() not in seen:
            seen.add(text.lower())
            out.append(text)
    return out


def to_dict_list(value, primary_key):
    """Any LLM answer -> list of dicts (plain strings become {primary_key: string})."""
    if value is None:
        return []
    if isinstance(value, dict):
        value = [value]
    if isinstance(value, str):
        value = [v for v in re.split(r"\n+", value) if v.strip()]
    out = []
    for item in value if isinstance(value, (list, tuple)) else [value]:
        if isinstance(item, dict):
            out.append({str(k): v for k, v in item.items()})
        elif item is not None and _clean(item):
            out.append({primary_key: _clean(item)})
    return out


def _to_score(value):
    m = re.search(r"\d+(\.\d+)?", str(value))
    return max(0, min(100, int(float(m.group())))) if m else 0


def _text(value, default=""):
    text = _clean(value) if value is not None else ""
    return text if text.lower() not in _EMPTY else default


def normalize_profile(raw):
    return {
        "full_name": _text(raw.get("full_name"), "Unknown"),
        "email": _text(raw.get("email"), "Unknown"),
        "education": to_dict_list(raw.get("education"), "degree"),
        "experience": to_dict_list(raw.get("experience"), "role"),
        "summary": _text(raw.get("summary")),
        "experience_level": _text(raw.get("experience_level"), "Fresher"),
    }


def normalize_skills(raw):
    return {
        "technical_skills": to_str_list(raw.get("technical_skills")),
        "tools_and_technologies": to_str_list(raw.get("tools_and_technologies")),
        "soft_skills": to_str_list(raw.get("soft_skills")),
        "domains": to_str_list(raw.get("domains")),
        "notable_projects": to_str_list(raw.get("notable_projects"))[:5],
    }


def normalize_gap(raw):
    order = {"High": 0, "Medium": 1, "Low": 2}
    missing = []
    for item in to_dict_list(raw.get("missing_skills"), "skill"):
        skill = _text(item.get("skill") or item.get("name"))
        if not skill:
            continue
        priority = _text(item.get("priority"), "Medium").capitalize()
        missing.append({
            "skill": skill,
            "priority": priority if priority in order else "Medium",
            "reason": _text(item.get("reason") or item.get("why")),
        })
    missing.sort(key=lambda m: order[m["priority"]])
    return {
        "matched_skills": to_str_list(raw.get("matched_skills")),
        "missing_skills": missing,
        "match_score": _to_score(raw.get("match_score")),
        "readiness_summary": _text(raw.get("readiness_summary")),
    }


def normalize_plan(raw):
    roadmap = []
    for i, p in enumerate(to_dict_list(raw.get("recommended_roadmap"), "phase"), 1):
        roadmap.append({
            "phase": _text(p.get("phase") or p.get("name") or p.get("title"), f"Phase {i}"),
            "duration": _text(p.get("duration")),
            "topics": to_str_list(p.get("topics")),
            "project": _text(p.get("project")),
        })
    jobs = []
    for j in to_dict_list(raw.get("suggested_jobs"), "title"):
        title = _text(j.get("title") or j.get("job_title") or j.get("role"))
        if title:
            jobs.append({"title": title,
                         "why_it_fits": _text(j.get("why_it_fits") or j.get("why")),
                         "next_step": _text(j.get("next_step"))})
    plan = []
    for p in to_dict_list(raw.get("improvement_plan"), "timeframe"):
        plan.append({"timeframe": _text(p.get("timeframe") or p.get("period"), "Next steps"),
                     "actions": to_str_list(p.get("actions") or p.get("steps"))})
    return {"recommended_roadmap": roadmap, "suggested_jobs": jobs, "improvement_plan": plan}


def read_cv_pdf(path):
    """Extract plain text from a CV PDF."""
    pages = PyPDFLoader(str(path)).load()
    return "\n".join(p.page_content for p in pages).strip()


## Step 6 - The four chains
Each chain is `prompt template | LLM + output parser` (LCEL). `generate_and_parse` retries once with a stricter instruction if the JSON is invalid, and `extract_json_block` is the lab's helper made robust (it no longer crashes when the model forgets the ```json fence).

In [41]:
def extract_json_block(text):
    """Robust version of the lab helper: accepts a ```json fence, or falls back to the outermost {...}."""
    blocks = re.findall(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    if blocks:
        body = blocks[-1]
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end <= start:
            raise ValueError("No JSON object found in the model output")
        body = text[start:end + 1]
    return f"```json\n{body}\n```"

def parse_lenient(parser, text):
    fenced = extract_json_block(text)
    inner = re.search(r"```json\s*(.*?)\s*```", fenced, re.DOTALL).group(1)
    data = json.loads(inner, strict=False)          # turn the text into a dict
    names = [s.name for s in parser.response_schemas]
    if not any(n in data for n in names):           # still reject answers that are totally wrong
        raise ValueError("None of the expected keys were found")
    for n in names:
        data.setdefault(n, None)                    # a missing key becomes None instead of an error
    return data
    
def generate_and_parse(prompt_text, parser, max_new_tokens=700, retries=1):
    """LLM call + output parser. If the JSON is invalid, ask once more with a stricter instruction."""
    last_error, raw = None, ""
    for attempt in range(retries + 1):
        suffix = "" if attempt == 0 else (
            "\n\nIMPORTANT: your previous answer was not valid JSON. "
            "Reply with ONE valid JSON object inside a ```json code block and nothing else.")
        raw = llm_generate(prompt_text + suffix, max_new_tokens)
        try:
            return parse_lenient(parser, raw)
        except Exception as error:                       # noqa: BLE001
            last_error = error
    raise ValueError(f"Could not parse the model output: {last_error}\n--- RAW OUTPUT ---\n{raw}")


def make_chain(template, parser, max_new_tokens=700):
    """LCEL chain:  prompt template  ->  LLM + output parser  ->  Python dict."""
    prompt = PromptTemplate.from_template(template).partial(
        format_instructions=parser.get_format_instructions())
    return prompt | RunnableLambda(
        lambda prompt_value: generate_and_parse(prompt_value.to_string(), parser, max_new_tokens))


ANALYZE_CV_TEMPLATE = """You are an expert HR recruiter and CV analyst.
Analyse the CV below and extract the candidate profile.
Use ONLY information found in the CV or the extra notes. If something is missing write "Unknown" or an empty list.

CV:
{cv_text}

Extra notes from the candidate about their experience:
{extra_info}

{format_instructions}
Return only the JSON code block."""

EXTRACT_SKILLS_TEMPLATE = """You are a technical recruiter who extracts skills from CVs.
Read the CV, the skills the candidate listed themselves and their experience notes.
Extract a complete, de-duplicated skill inventory. Use short, standard skill names (for example "PyTorch", not "I used PyTorch to train models").
Do not invent skills that are not supported by the text.

CV:
{cv_text}

Skills listed by the candidate:
{stated_skills}

Experience notes:
{extra_info}

{format_instructions}
Return only the JSON code block."""

COMPARE_TEMPLATE = """You are a career coach comparing a candidate with a target job.

Target role: {target_role}

Candidate skills (JSON):
{candidate_skills}

Job requirements from the knowledge base:
{job_context}

Industry requirements from the knowledge base:
{industry_context}

Instructions:
- matched_skills: skills the candidate HAS that the role requires.
- missing_skills: skills the role requires that the candidate does NOT show. Use only skills mentioned in the requirements above, most important first. Priority High = required for the role, Medium = expected, Low = nice to have.
- match_score: honest readiness score from 0 to 100.
- Never claim the candidate has a skill that is not in their skill list.

{format_instructions}
Return only the JSON code block."""

PLAN_TEMPLATE = """You are an experienced career coach. Write a practical plan for the candidate.

Target role: {target_role}
Candidate level: {experience_level}
Candidate summary: {summary}
Current skills: {current_skills}
Missing skills (most important first): {missing_skills}
Readiness score: {match_score} / 100

Reference learning roadmap:
{roadmap_context}

Industry expectations:
{industry_context}

Other roles that match the candidate's skills (from the job knowledge base):
{related_roles}

Instructions:
- recommended_roadmap: ordered phases that close the missing skills. Adapt the reference roadmap and skip what the candidate already knows.
- suggested_jobs: exactly 3 realistic job titles. Include the target role (as a stretch goal if the score is low) and the best-matching alternatives.
- improvement_plan: three timeframes (30, 60 and 90 days) with concrete actions such as projects, portfolio, certificates and CV improvements.

{format_instructions}
Return only the JSON code block."""

analyze_cv_chain = make_chain(ANALYZE_CV_TEMPLATE, profile_parser, max_new_tokens=700)
extract_skills_chain = make_chain(EXTRACT_SKILLS_TEMPLATE, skills_parser, max_new_tokens=700)
compare_chain = make_chain(COMPARE_TEMPLATE, gap_parser, max_new_tokens=800)
plan_chain = make_chain(PLAN_TEMPLATE, plan_parser, max_new_tokens=1400)


## Step 7 - Full pipeline
`run_career_coach` connects everything: chain 1 and 2 on the user input, RAG for job descriptions and industry requirements, chain 3 to find the gap, RAG for the roadmap, chain 4 for the plan, and finally the five requested sections are assembled into one report dict.

In [42]:
def _json(data):
    return json.dumps(data, ensure_ascii=False, indent=1)


def run_career_coach(cv_text, skills_text="", experience_text="", target_role="", progress=None):
    """Full pipeline: CV + skills + experience + target role  ->  structured career report."""
    say = progress or print
    cv_text = (cv_text or "").strip()[:MAX_CV_CHARS]
    skills_text = (skills_text or "").strip()
    experience_text = (experience_text or "").strip()
    if not cv_text and not skills_text:
        raise ValueError("Please provide a CV or at least a list of skills.")

    # Chain 1 - analyse the CV
    say("Step 1/4 - Analysing the CV ...")
    profile = normalize_profile(analyze_cv_chain.invoke({
        "cv_text": cv_text or "Not provided",
        "extra_info": experience_text or "None"}))

    # Chain 2 - extract skills
    say("Step 2/4 - Extracting skills ...")
    skills = normalize_skills(extract_skills_chain.invoke({
        "cv_text": cv_text or "Not provided",
        "stated_skills": skills_text or "None",
        "extra_info": experience_text or "None"}))
    hard_skills = skills["technical_skills"] + skills["tools_and_technologies"]
    skills_query = ", ".join(hard_skills[:25]) or skills_text

    # Pick the knowledge-base role that matches the target (or the best fit if none was given)
    requested = (target_role or "").strip()
    kb_role = resolve_role(requested) if requested else \
        best_fit_roles(f"{skills_query}. {profile['summary']}", k=1)[0]

    # RAG (job descriptions + industry requirements) -> Chain 3 - compare with job requirements
    say(f"Step 3/4 - Retrieving requirements for '{kb_role}' and comparing ...")
    job_docs = retrieve(f"{kb_role} required skills", "job_description", role=kb_role, k=3)
    industry_docs = retrieve(f"{kb_role} industry requirements", "industry_requirement", role=kb_role, k=3)
    gap = normalize_gap(compare_chain.invoke({
        "target_role": kb_role,
        "candidate_skills": _json(skills),
        "job_context": format_docs(job_docs),
        "industry_context": format_docs(industry_docs)}))

    # RAG (learning roadmaps) -> Chain 4 - roadmap, suggested jobs, improvement plan
    say("Step 4/4 - Building the roadmap and improvement plan ...")
    roadmap_docs = retrieve(f"{kb_role} learning roadmap", "learning_roadmap", role=kb_role, k=3)
    related = best_fit_roles(skills_query or kb_role, k=4)
    plan = normalize_plan(plan_chain.invoke({
        "target_role": kb_role,
        "experience_level": profile["experience_level"],
        "summary": profile["summary"] or "Not available",
        "current_skills": ", ".join(hard_skills) or "None listed",
        "missing_skills": ", ".join(m["skill"] for m in gap["missing_skills"]) or "None identified",
        "match_score": gap["match_score"],
        "roadmap_context": format_docs(roadmap_docs),
        "industry_context": format_docs(industry_docs),
        "related_roles": "\n".join(role_snapshot(r) for r in related)}))

    report = {
        "target_role": requested or kb_role,
        "kb_role": kb_role,
        "candidate": {"name": profile["full_name"], "level": profile["experience_level"],
                      "summary": profile["summary"]},
        "match_score": gap["match_score"],
        "readiness_summary": gap["readiness_summary"],
        "current_skills": {"technical": skills["technical_skills"],
                           "tools": skills["tools_and_technologies"],
                           "soft": skills["soft_skills"]},
        "matched_skills": gap["matched_skills"],
        "missing_skills": gap["missing_skills"],
        "recommended_roadmap": plan["recommended_roadmap"],
        "suggested_jobs": plan["suggested_jobs"],
        "improvement_plan": plan["improvement_plan"],
    }
    sources = [{"role": d.metadata["role"], "type": d.metadata["doc_type"], "file": d.metadata["source"]}
               for d in job_docs + industry_docs + roadmap_docs]
    return {"report": report, "profile": profile, "skills": skills, "gap": gap, "sources": sources}


### Try it on your CV
Edit the skills, experience and target role. Leave `target_role=""` to let the coach pick the best-fitting role. This runs 4 LLM calls, so expect a few minutes on a T4.

In [43]:
cv_text = read_cv_pdf(pdf_path)          # pdf_path comes from the CV cell at the top of this notebook

result = run_career_coach(
    cv_text=cv_text,
    skills_text="Python, SQL, PyTorch, LangChain",              # <- edit me
    experience_text="AI internship, graduation project",       # <- edit me
    target_role="AI / LLM Engineer",                           # <- edit me (or "")
)

Step 1/4 - Analysing the CV ...
Step 2/4 - Extracting skills ...
Step 3/4 - Retrieving requirements for 'AI / LLM Engineer' and comparing ...
Step 4/4 - Building the roadmap and improvement plan ...


In [44]:
def print_report(report):
    print(f"TARGET ROLE : {report['target_role']}   |   READINESS: {report['match_score']}/100")
    print(report["readiness_summary"])
    print("\nCURRENT SKILLS")
    for group, items in report["current_skills"].items():
        print(f"  {group:9}: {', '.join(items)}")
    print("\nMISSING SKILLS")
    for m in report["missing_skills"]:
        print(f"  [{m['priority']:6}] {m['skill']} - {m['reason']}")
    print("\nRECOMMENDED ROADMAP")
    for i, p in enumerate(report["recommended_roadmap"], 1):
        print(f"  {i}. {p['phase']} ({p['duration']}): {', '.join(p['topics'])}  -> project: {p['project']}")
    print("\nSUGGESTED JOBS")
    for j in report["suggested_jobs"]:
        print(f"  - {j['title']}: {j['why_it_fits']}  (next: {j['next_step']})")
    print("\nIMPROVEMENT PLAN")
    for s in report["improvement_plan"]:
        print(f"  {s['timeframe']}:")
        for a in s["actions"]:
            print(f"     * {a}")

print_report(result["report"])

TARGET ROLE : AI / LLM Engineer   |   READINESS: 75/100
The candidate has a solid foundation with 8 out of the 13 required skills. They lack experience with fine-tuning models, evaluation frameworks, agents and tool calling, FastAPI, Docker, and quantisation. With some focused learning and project experience, they could be ready for the role in a few months.

CURRENT SKILLS
  technical: Python, SQL, PyTorch, TensorFlow, Deep Learning, NLP, Computer Vision, Transformers, CNN, RNN, Transfer Learning, Model Evaluation, Optimization, Git, GitHub, Jupyter, Google Colab, Linux, MySQL, VS Code, Data Structures & Algorithms, Object-Oriented Programming (OOP), Database Systems, Operating Systems, Computer Organization, Computer Networks, Cybersecurity Fundamentals, System Programming, Software Engineering, Data Analysis, Data Modeling, Machine Learning Algorithms, SQL Databases, System Design, Secure Application Development, Data Processing, Software Optimization, Fortinet Security Fabric, Wire

In [45]:
import json
# Intermediate outputs of each chain (useful for debugging prompts)
print(json.dumps({"profile": result["profile"], "skills": result["skills"], "gap": result["gap"]}, indent=2)[:3000])
print("\nRAG sources:", result["sources"])

{
  "profile": {
    "full_name": "Basem Hesham Rezk",
    "email": "basemhesham200318@gmail.com",
    "education": [
      {
        "degree": "Bachelor of computer and communication Engineering (Minor Ai)",
        "institution": "Alexandria University, Egypt",
        "year": "2026"
      }
    ],
    "experience": [
      {
        "role": "Teaching Assistant",
        "company": "Alexandria National University",
        "years": "Sep 2026 \u2013 Present"
      },
      {
        "role": "Natural Language Processing Intern",
        "company": "Tips Hindawy (Edrak)",
        "years": "Aug 2026 \u2013 Sep 2026"
      },
      {
        "role": "Natural Language Processing Intern",
        "company": "Cellula Technologies",
        "years": "Jul 2026 \u2013 Sep 2026"
      },
      {
        "role": "Backend Developer Intern (Laravel)",
        "company": "RMZtech",
        "years": "Jul 2025 \u2013 Aug 2025"
      },
      {
        "role": "Fortinet Cybersecurity Training",
       

---
# Part 3 - Streamlit app
The app needs two files next to each other: `career_coach.py` (all the code from Part 2, as a module) and `app.py` (the UI). The next cells write both files into `/kaggle/working`, so you can also download them and run the app on your own machine.

In [65]:
%%writefile career_coach.py
"""career_coach.py - RAG + chains + output parsers for the AI Career Coach.
Generated from the notebook; imported by app.py (Streamlit)."""

import os
import re
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------
BASE_DIR = Path(os.environ.get(
    "COACH_BASE_DIR", "/kaggle/working" if os.path.exists("/kaggle/working") else "."))
KB_DIR = BASE_DIR / "knowledge_base"          # job descriptions / roadmaps / industry docs
INDEX_DIR = BASE_DIR / "faiss_career_index"   # saved FAISS index
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_NAME = os.environ.get("COACH_MODEL", "mistralai/Mistral-7B-Instruct-v0.2")
MAX_CV_CHARS = 6000                           # keep prompts short enough for a 7B model

# folder name -> metadata value used for filtered retrieval
DOC_TYPES = {
    "job_descriptions": "job_description",
    "learning_roadmaps": "learning_roadmap",
    "industry_requirements": "industry_requirement",
}


# Filled by load_llm() (inside the notebook they are the model/tokenizer loaded earlier)
tokenizer = None
model = None


def load_llm(model_name=MODEL_NAME, use_4bit=False):
    """Load tokenizer + model into the module globals `tokenizer` / `model`.
    (In the notebook the model is already loaded, so this is only used by the Streamlit app.)"""
    global tokenizer, model
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    kwargs = {"device_map": "auto"}
    if use_4bit and torch.cuda.is_available():
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    else:
        kwargs["torch_dtype"] = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    return model, tokenizer


def llm_generate(prompt, max_new_tokens=700):
    """Greedy generation that returns ONLY the new text (the lab's generate_text echoed the prompt too).
    Greedy decoding keeps JSON output stable and repeatable."""
    import torch
    if model is None:
        load_llm()
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def load_kb_documents(kb_dir=KB_DIR):
    """Read every .md / .txt / .pdf under knowledge_base/<doc-type folder>/ into LangChain Documents.
    A file may start with a line 'Role: <role name>'; otherwise the file name is used as the role."""
    docs = []
    for folder, doc_type in DOC_TYPES.items():
        for path in sorted((Path(kb_dir) / folder).glob("*")):
            suffix = path.suffix.lower()
            if suffix in {".md", ".txt"}:
                text = path.read_text(encoding="utf-8")
            elif suffix == ".pdf":
                text = "\n".join(p.page_content for p in PyPDFLoader(str(path)).load())
            else:
                continue
            match = re.search(r"^Role:\s*(.+)$", text, re.MULTILINE)
            role = match.group(1).strip() if match else path.stem.replace("_", " ").title()
            docs.append(Document(
                page_content=text,
                metadata={"role": role, "doc_type": doc_type, "source": path.name},
            ))
    return docs


def list_roles(kb_dir=KB_DIR):
    """All roles that have a job description in the knowledge base."""
    return sorted({d.metadata["role"] for d in load_kb_documents(kb_dir)
                   if d.metadata["doc_type"] == "job_description"})


_VECTORSTORE = None


def get_vectorstore(rebuild=False):
    """Load the FAISS index from disk (or build it from the knowledge base)."""
    global _VECTORSTORE
    if _VECTORSTORE is not None and not rebuild:
        return _VECTORSTORE
    embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cpu"})
    if INDEX_DIR.exists() and not rebuild:
        _VECTORSTORE = FAISS.load_local(
            str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
        return _VECTORSTORE
    docs = load_kb_documents()
    if not docs:
        raise FileNotFoundError(
            f"No knowledge-base files found in {KB_DIR}. Run the 'Build the knowledge base' "
            "cells in the notebook first.")
    splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    _VECTORSTORE = FAISS.from_documents(chunks, embeddings)
    _VECTORSTORE.save_local(str(INDEX_DIR))
    return _VECTORSTORE


def retrieve(query, doc_type, role=None, k=4):
    """Semantic search restricted to one document type (and optionally one role)."""
    metadata_filter = {"doc_type": doc_type}
    if role:
        metadata_filter["role"] = role
    return get_vectorstore().similarity_search(query, k=k, filter=metadata_filter, fetch_k=500)


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['role']} | {d.metadata['doc_type']}]\n{d.page_content}"
                       for d in docs)


def resolve_role(text):
    """Map free text ('senior data scientist', 'NLP engineer') to the closest role in the knowledge base."""
    roles = list_roles()
    wanted = text.strip().lower()
    for role in roles:                                   # exact match
        if role.lower() == wanted:
            return role
    for role in roles:                                   # containment match
        if len(wanted) >= 4 and (role.lower() in wanted or wanted in role.lower()):
            return role
    hit = get_vectorstore().similarity_search(           # semantic fallback
        text, k=1, filter={"doc_type": "job_description"}, fetch_k=500)
    return hit[0].metadata["role"]


def best_fit_roles(query, k=3):
    """Roles whose job descriptions are semantically closest to a skills description."""
    docs = get_vectorstore().similarity_search(
        query, k=30, filter={"doc_type": "job_description"}, fetch_k=500)
    roles = []
    for d in docs:
        if d.metadata["role"] not in roles:
            roles.append(d.metadata["role"])
    return roles[:k]


def role_snapshot(role):
    """One-line description of a role, taken from its job description."""
    docs = retrieve(f"{role} summary", "job_description", role=role, k=1)
    if not docs:
        return role
    m = re.search(r"Summary:\s*(.+)", docs[0].page_content)
    return f"{role}: {m.group(1).strip() if m else docs[0].page_content[:200]}"


def make_parser(fields):
    """Build a StructuredOutputParser from (name, description) pairs."""
    return StructuredOutputParser.from_response_schemas(
        [ResponseSchema(name=n, description=d) for n, d in fields])


# Parser 1 - CV analysis (extends the lab's original schema)
profile_parser = make_parser([
    ("full_name", "The candidate's full name as a string, or \"Unknown\""),
    ("email", "The candidate's email address as a string, or \"Unknown\""),
    ("education", "List of objects, each with the keys degree, institution, year"),
    ("experience", "List of objects, each with the keys role, company, years. Include internships and training programs"),
    ("summary", "Two-sentence professional summary of the candidate, as a string"),
    ("experience_level", "Exactly one of: Student, Fresher, Junior, Mid-level, Senior"),
])

# Parser 2 - skill extraction
skills_parser = make_parser([
    ("technical_skills", "List of strings: programming languages and technical concepts (e.g. Python, SQL, machine learning, NLP)"),
    ("tools_and_technologies", "List of strings: frameworks, libraries, platforms and tools (e.g. PyTorch, Docker, LangChain)"),
    ("soft_skills", "List of strings: soft skills such as communication or teamwork"),
    ("domains", "List of strings: fields the candidate has worked or studied in"),
    ("notable_projects", "List of at most 5 short strings, one per project"),
])

# Parser 3 - CV vs. job requirements
gap_parser = make_parser([
    ("matched_skills", "List of strings: candidate skills that the target role requires"),
    ("missing_skills", "List of objects, each with the keys skill, priority (High, Medium or Low) and reason (one short sentence)"),
    ("match_score", "Integer from 0 to 100: how ready the candidate is for the target role today"),
    ("readiness_summary", "Two or three honest sentences explaining the score"),
])

# Parser 4 - roadmap, jobs and improvement plan
plan_parser = make_parser([
    ("recommended_roadmap", "List of 4 to 6 phases, each an object with the keys phase (short name), duration (e.g. '4 weeks'), topics (list of strings) and project (one practical project)"),
    ("suggested_jobs", "List of exactly 3 objects, each with the keys title, why_it_fits and next_step"),
    ("improvement_plan", "List of 3 objects, each with the keys timeframe (e.g. 'Next 30 days') and actions (list of 2 to 4 concrete strings)"),
])


_EMPTY = {"", "none", "n/a", "unknown", "not provided", "null"}


def _clean(value):
    return re.sub(r"\s+", " ", str(value)).strip()


def _flatten(value):
    if value is None:
        return []
    if isinstance(value, dict):
        name = value.get("skill") or value.get("name") or value.get("title")
        return [name] if name else [x for v in value.values() for x in _flatten(v)]
    if isinstance(value, (list, tuple, set)):
        return [x for v in value for x in _flatten(v)]
    return [value]


def to_str_list(value):
    """Any LLM answer -> clean, de-duplicated list of strings."""
    if isinstance(value, str):
        value = re.split(r"[\n,;\u2022]+", value)
    out, seen = [], set()
    for item in _flatten(value):
        text = _clean(item).strip("-*\u2022 ")
        if text.lower() not in _EMPTY and text.lower() not in seen:
            seen.add(text.lower())
            out.append(text)
    return out


def to_dict_list(value, primary_key):
    """Any LLM answer -> list of dicts (plain strings become {primary_key: string})."""
    if value is None:
        return []
    if isinstance(value, dict):
        value = [value]
    if isinstance(value, str):
        value = [v for v in re.split(r"\n+", value) if v.strip()]
    out = []
    for item in value if isinstance(value, (list, tuple)) else [value]:
        if isinstance(item, dict):
            out.append({str(k): v for k, v in item.items()})
        elif item is not None and _clean(item):
            out.append({primary_key: _clean(item)})
    return out


def _to_score(value):
    m = re.search(r"\d+(\.\d+)?", str(value))
    return max(0, min(100, int(float(m.group())))) if m else 0


def _text(value, default=""):
    text = _clean(value) if value is not None else ""
    return text if text.lower() not in _EMPTY else default


def normalize_profile(raw):
    return {
        "full_name": _text(raw.get("full_name"), "Unknown"),
        "email": _text(raw.get("email"), "Unknown"),
        "education": to_dict_list(raw.get("education"), "degree"),
        "experience": to_dict_list(raw.get("experience"), "role"),
        "summary": _text(raw.get("summary")),
        "experience_level": _text(raw.get("experience_level"), "Fresher"),
    }


def normalize_skills(raw):
    return {
        "technical_skills": to_str_list(raw.get("technical_skills")),
        "tools_and_technologies": to_str_list(raw.get("tools_and_technologies")),
        "soft_skills": to_str_list(raw.get("soft_skills")),
        "domains": to_str_list(raw.get("domains")),
        "notable_projects": to_str_list(raw.get("notable_projects"))[:5],
    }


def normalize_gap(raw):
    order = {"High": 0, "Medium": 1, "Low": 2}
    missing = []
    for item in to_dict_list(raw.get("missing_skills"), "skill"):
        skill = _text(item.get("skill") or item.get("name"))
        if not skill:
            continue
        priority = _text(item.get("priority"), "Medium").capitalize()
        missing.append({
            "skill": skill,
            "priority": priority if priority in order else "Medium",
            "reason": _text(item.get("reason") or item.get("why")),
        })
    missing.sort(key=lambda m: order[m["priority"]])
    return {
        "matched_skills": to_str_list(raw.get("matched_skills")),
        "missing_skills": missing,
        "match_score": _to_score(raw.get("match_score")),
        "readiness_summary": _text(raw.get("readiness_summary")),
    }


def normalize_plan(raw):
    roadmap = []
    for i, p in enumerate(to_dict_list(raw.get("recommended_roadmap"), "phase"), 1):
        roadmap.append({
            "phase": _text(p.get("phase") or p.get("name") or p.get("title"), f"Phase {i}"),
            "duration": _text(p.get("duration")),
            "topics": to_str_list(p.get("topics")),
            "project": _text(p.get("project")),
        })
    jobs = []
    for j in to_dict_list(raw.get("suggested_jobs"), "title"):
        title = _text(j.get("title") or j.get("job_title") or j.get("role"))
        if title:
            jobs.append({"title": title,
                         "why_it_fits": _text(j.get("why_it_fits") or j.get("why")),
                         "next_step": _text(j.get("next_step"))})
    plan = []
    for p in to_dict_list(raw.get("improvement_plan"), "timeframe"):
        plan.append({"timeframe": _text(p.get("timeframe") or p.get("period"), "Next steps"),
                     "actions": to_str_list(p.get("actions") or p.get("steps"))})
    return {"recommended_roadmap": roadmap, "suggested_jobs": jobs, "improvement_plan": plan}


def read_cv_pdf(path):
    """Extract plain text from a CV PDF."""
    pages = PyPDFLoader(str(path)).load()
    return "\n".join(p.page_content for p in pages).strip()


def extract_json_block(text):
    """Robust version of the lab helper: accepts a ```json fence, or falls back to the outermost {...}."""
    blocks = re.findall(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    if blocks:
        body = blocks[-1]
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end <= start:
            raise ValueError("No JSON object found in the model output")
        body = text[start:end + 1]
    return f"```json\n{body}\n```"

def parse_lenient(parser, text):
    fenced = extract_json_block(text)
    inner = re.search(r"```json\s*(.*?)\s*```", fenced, re.DOTALL).group(1)
    data = json.loads(inner, strict=False)          # turn the text into a dict
    names = [s.name for s in parser.response_schemas]
    if not any(n in data for n in names):           # still reject answers that are totally wrong
        raise ValueError("None of the expected keys were found")
    for n in names:
        data.setdefault(n, None)                    # a missing key becomes None instead of an error
    return data
    
def generate_and_parse(prompt_text, parser, max_new_tokens=700, retries=1):
    """LLM call + output parser. If the JSON is invalid, ask once more with a stricter instruction."""
    last_error, raw = None, ""
    for attempt in range(retries + 1):
        suffix = "" if attempt == 0 else (
            "\n\nIMPORTANT: your previous answer was not valid JSON. "
            "Reply with ONE valid JSON object inside a ```json code block and nothing else.")
        raw = llm_generate(prompt_text + suffix, max_new_tokens)
        try:
            return parse_lenient(parser, raw)
        except Exception as error:                       # noqa: BLE001
            last_error = error
    raise ValueError(f"Could not parse the model output: {last_error}\n--- RAW OUTPUT ---\n{raw}")


def make_chain(template, parser, max_new_tokens=700):
    """LCEL chain:  prompt template  ->  LLM + output parser  ->  Python dict."""
    prompt = PromptTemplate.from_template(template).partial(
        format_instructions=parser.get_format_instructions())
    return prompt | RunnableLambda(
        lambda prompt_value: generate_and_parse(prompt_value.to_string(), parser, max_new_tokens))


ANALYZE_CV_TEMPLATE = """You are an expert HR recruiter and CV analyst.
Analyse the CV below and extract the candidate profile.
Use ONLY information found in the CV or the extra notes. If something is missing write "Unknown" or an empty list.

CV:
{cv_text}

Extra notes from the candidate about their experience:
{extra_info}

{format_instructions}
Return only the JSON code block."""

EXTRACT_SKILLS_TEMPLATE = """You are a technical recruiter who extracts skills from CVs.
Read the CV, the skills the candidate listed themselves and their experience notes.
Extract a complete, de-duplicated skill inventory. Use short, standard skill names (for example "PyTorch", not "I used PyTorch to train models").
Do not invent skills that are not supported by the text.

CV:
{cv_text}

Skills listed by the candidate:
{stated_skills}

Experience notes:
{extra_info}

{format_instructions}
Return only the JSON code block."""

COMPARE_TEMPLATE = """You are a career coach comparing a candidate with a target job.

Target role: {target_role}

Candidate skills (JSON):
{candidate_skills}

Job requirements from the knowledge base:
{job_context}

Industry requirements from the knowledge base:
{industry_context}

Instructions:
- matched_skills: skills the candidate HAS that the role requires.
- missing_skills: skills the role requires that the candidate does NOT show. Use only skills mentioned in the requirements above, most important first. Priority High = required for the role, Medium = expected, Low = nice to have.
- match_score: honest readiness score from 0 to 100.
- Never claim the candidate has a skill that is not in their skill list.

{format_instructions}
Return only the JSON code block."""

PLAN_TEMPLATE = """You are an experienced career coach. Write a practical plan for the candidate.

Target role: {target_role}
Candidate level: {experience_level}
Candidate summary: {summary}
Current skills: {current_skills}
Missing skills (most important first): {missing_skills}
Readiness score: {match_score} / 100

Reference learning roadmap:
{roadmap_context}

Industry expectations:
{industry_context}

Other roles that match the candidate's skills (from the job knowledge base):
{related_roles}

Instructions:
- recommended_roadmap: ordered phases that close the missing skills. Adapt the reference roadmap and skip what the candidate already knows.
- suggested_jobs: exactly 3 realistic job titles. Include the target role (as a stretch goal if the score is low) and the best-matching alternatives.
- improvement_plan: three timeframes (30, 60 and 90 days) with concrete actions such as projects, portfolio, certificates and CV improvements.

{format_instructions}
Return only the JSON code block."""

analyze_cv_chain = make_chain(ANALYZE_CV_TEMPLATE, profile_parser, max_new_tokens=700)
extract_skills_chain = make_chain(EXTRACT_SKILLS_TEMPLATE, skills_parser, max_new_tokens=700)
compare_chain = make_chain(COMPARE_TEMPLATE, gap_parser, max_new_tokens=800)
plan_chain = make_chain(PLAN_TEMPLATE, plan_parser, max_new_tokens=1400)


def _json(data):
    return json.dumps(data, ensure_ascii=False, indent=1)


def run_career_coach(cv_text, skills_text="", experience_text="", target_role="", progress=None):
    """Full pipeline: CV + skills + experience + target role  ->  structured career report."""
    say = progress or print
    cv_text = (cv_text or "").strip()[:MAX_CV_CHARS]
    skills_text = (skills_text or "").strip()
    experience_text = (experience_text or "").strip()
    if not cv_text and not skills_text:
        raise ValueError("Please provide a CV or at least a list of skills.")

    # Chain 1 - analyse the CV
    say("Step 1/4 - Analysing the CV ...")
    profile = normalize_profile(analyze_cv_chain.invoke({
        "cv_text": cv_text or "Not provided",
        "extra_info": experience_text or "None"}))

    # Chain 2 - extract skills
    say("Step 2/4 - Extracting skills ...")
    skills = normalize_skills(extract_skills_chain.invoke({
        "cv_text": cv_text or "Not provided",
        "stated_skills": skills_text or "None",
        "extra_info": experience_text or "None"}))
    hard_skills = skills["technical_skills"] + skills["tools_and_technologies"]
    skills_query = ", ".join(hard_skills[:25]) or skills_text

    # Pick the knowledge-base role that matches the target (or the best fit if none was given)
    requested = (target_role or "").strip()
    kb_role = resolve_role(requested) if requested else \
        best_fit_roles(f"{skills_query}. {profile['summary']}", k=1)[0]

    # RAG (job descriptions + industry requirements) -> Chain 3 - compare with job requirements
    say(f"Step 3/4 - Retrieving requirements for '{kb_role}' and comparing ...")
    job_docs = retrieve(f"{kb_role} required skills", "job_description", role=kb_role, k=3)
    industry_docs = retrieve(f"{kb_role} industry requirements", "industry_requirement", role=kb_role, k=3)
    gap = normalize_gap(compare_chain.invoke({
        "target_role": kb_role,
        "candidate_skills": _json(skills),
        "job_context": format_docs(job_docs),
        "industry_context": format_docs(industry_docs)}))

    # RAG (learning roadmaps) -> Chain 4 - roadmap, suggested jobs, improvement plan
    say("Step 4/4 - Building the roadmap and improvement plan ...")
    roadmap_docs = retrieve(f"{kb_role} learning roadmap", "learning_roadmap", role=kb_role, k=3)
    related = best_fit_roles(skills_query or kb_role, k=4)
    plan = normalize_plan(plan_chain.invoke({
        "target_role": kb_role,
        "experience_level": profile["experience_level"],
        "summary": profile["summary"] or "Not available",
        "current_skills": ", ".join(hard_skills) or "None listed",
        "missing_skills": ", ".join(m["skill"] for m in gap["missing_skills"]) or "None identified",
        "match_score": gap["match_score"],
        "roadmap_context": format_docs(roadmap_docs),
        "industry_context": format_docs(industry_docs),
        "related_roles": "\n".join(role_snapshot(r) for r in related)}))

    report = {
        "target_role": requested or kb_role,
        "kb_role": kb_role,
        "candidate": {"name": profile["full_name"], "level": profile["experience_level"],
                      "summary": profile["summary"]},
        "match_score": gap["match_score"],
        "readiness_summary": gap["readiness_summary"],
        "current_skills": {"technical": skills["technical_skills"],
                           "tools": skills["tools_and_technologies"],
                           "soft": skills["soft_skills"]},
        "matched_skills": gap["matched_skills"],
        "missing_skills": gap["missing_skills"],
        "recommended_roadmap": plan["recommended_roadmap"],
        "suggested_jobs": plan["suggested_jobs"],
        "improvement_plan": plan["improvement_plan"],
    }
    sources = [{"role": d.metadata["role"], "type": d.metadata["doc_type"], "file": d.metadata["source"]}
               for d in job_docs + industry_docs + roadmap_docs]
    return {"report": report, "profile": profile, "skills": skills, "gap": gap, "sources": sources}


Overwriting career_coach.py


In [66]:
%%writefile app.py
"""Simple Streamlit UI for the AI Career Coach.   Run:  streamlit run app.py"""
import json
import os
import tempfile

import pandas as pd
import streamlit as st

import career_coach as cc

st.set_page_config(page_title="AI Career Coach", page_icon="🧭", layout="wide")

NOT_SURE = "Not sure - suggest the best fit for me"
OTHER = "Other (type it below)"


@st.cache_resource(show_spinner="Loading the language model and knowledge base (first run takes a few minutes)...")
def init_backend():
    """Load the LLM and the FAISS index once per server process."""
    cc.load_llm(use_4bit=os.environ.get("COACH_4BIT", "1") == "1")
    cc.get_vectorstore()
    return cc.list_roles()


def render_report(result):
    r = result["report"]

    st.subheader(f"Target role: {r['target_role']}")
    if r["target_role"].lower() != r["kb_role"].lower():
        st.caption(f"Closest role in the knowledge base: **{r['kb_role']}**")
    left, right = st.columns([1, 3])
    left.metric("Estimated readiness", f"{r['match_score']} / 100")
    right.progress(r["match_score"] / 100)
    right.write(r["readiness_summary"])
    st.caption("The score is an LLM estimate based on retrieved job requirements, not an exact measurement.")

    tabs = st.tabs(["✅ Current skills", "⚠️ Missing skills", "🗺️ Roadmap",
                    "💼 Suggested jobs", "📈 Improvement plan", "🔍 Under the hood"])

    with tabs[0]:
        for label, key in [("Technical", "technical"), ("Tools & technologies", "tools"), ("Soft skills", "soft")]:
            st.markdown(f"**{label}**")
            items = r["current_skills"][key]
            st.write(" · ".join(f"`{s}`" for s in items) if items else "_None found_")
        if r["matched_skills"]:
            st.markdown("**Already relevant for this role**")
            st.write(" · ".join(f"`{s}`" for s in r["matched_skills"]))

    with tabs[1]:
        if r["missing_skills"]:
            df = pd.DataFrame(r["missing_skills"]).rename(
                columns={"skill": "Skill", "priority": "Priority", "reason": "Why it matters"})
            st.dataframe(df, hide_index=True)
        else:
            st.success("No major gaps were found for this role.")

    with tabs[2]:
        for i, p in enumerate(r["recommended_roadmap"], 1):
            with st.expander(f"{i}. {p['phase']}  ·  {p['duration']}", expanded=(i == 1)):
                for t in p["topics"]:
                    st.markdown(f"- {t}")
                if p["project"]:
                    st.markdown(f"**Project:** {p['project']}")

    with tabs[3]:
        for j in r["suggested_jobs"]:
            st.markdown(f"#### {j['title']}")
            st.write(j["why_it_fits"])
            if j["next_step"]:
                st.markdown(f"**Next step:** {j['next_step']}")
            st.divider()

    with tabs[4]:
        for step in r["improvement_plan"]:
            st.markdown(f"#### {step['timeframe']}")
            for a in step["actions"]:
                st.markdown(f"- {a}")

    with tabs[5]:
        st.caption("Intermediate results of each chain, and the knowledge-base documents that were retrieved.")
        with st.expander("Chain 1 - CV analysis"):
            st.json(result["profile"])
        with st.expander("Chain 2 - Extracted skills"):
            st.json(result["skills"])
        with st.expander("Chain 3 - Comparison with job requirements"):
            st.json(result["gap"])
        with st.expander("RAG sources used"):
            st.dataframe(pd.DataFrame(result["sources"]), hide_index=True)
        st.download_button("Download full report (JSON)",
                           json.dumps(result["report"], indent=2, ensure_ascii=False),
                           file_name="career_report.json", mime="application/json")


# ----------------------------------------------------------------------------- UI
st.title("🧭 AI Career Coach")
st.write("Upload your CV, add your skills and target role, and get a personalised gap analysis, "
         "learning roadmap, suggested jobs and improvement plan.")

try:
    roles = init_backend()
except Exception as error:  # noqa: BLE001
    st.error(f"Could not start the backend: {error}")
    st.stop()

with st.form("career_form"):
    col1, col2 = st.columns(2)
    with col1:
        cv_file = st.file_uploader("CV (PDF)", type=["pdf"])
        cv_pasted = st.text_area("...or paste your CV text", height=150)
        skills_text = st.text_area("Your skills (comma separated)", height=90,
                                   placeholder="Python, SQL, PyTorch, LangChain ...")
    with col2:
        experience_text = st.text_area("Experience (internships, projects, years of work)", height=150)
        choice = st.selectbox("Target role", [NOT_SURE] + roles + [OTHER])
        custom_role = st.text_input("Custom target role", disabled=(choice != OTHER))
    submitted = st.form_submit_button("Analyse my career path", type="primary")

if submitted:
    cv_text = cv_pasted
    if cv_file is not None:
        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
            tmp.write(cv_file.getvalue())
        try:
            cv_text = cc.read_cv_pdf(tmp.name)
        finally:
            os.unlink(tmp.name)
        if not cv_text:
            st.warning("No text could be read from this PDF (is it a scan?). Paste the text instead.")

    target = "" if choice == NOT_SURE else (custom_role if choice == OTHER else choice)

    try:
        with st.status("Working on your career report...", expanded=True) as status:
            st.session_state["result"] = cc.run_career_coach(
                cv_text, skills_text, experience_text, target, progress=st.write)
            status.update(label="Done!", state="complete", expanded=False)
    except Exception as error:  # noqa: BLE001
        st.session_state.pop("result", None)
        st.error(str(error))

if "result" in st.session_state:
    render_report(st.session_state["result"])


Overwriting app.py


In [71]:
%%writefile requirements.txt
streamlit
pandas
torch
transformers==4.52.4
accelerate
bitsandbytes
sentence-transformers
faiss-cpu
pypdf
langchain
langchain-core
langchain-community
langchain-classic
langchain-huggingface
langchain-text-splitters

Overwriting requirements.txt


3KTMNxJ9ClPVY0cjLeSzRdNUyBn_71nsmgJDwg2rmmRSSmyJQ

In [56]:
pip install fastapi uvicorn pyngrok  accelerate -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [59]:
NGROK_TOKEN = "3KTMNxJ9ClPVY0cjLeSzRdNUyBn_71nsmgJDwg2rmmRSSmyJQ"
API_KEY = "secret123"

In [62]:
import gc, torch
for name in ("model", "tokenizer"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()
print("GPU memory in use (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))

GPU memory in use (GB): 0.01


In [69]:
import os, subprocess, time
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
os.environ["COACH_4BIT"] = "1"        # set to "0" for full fp16 (needs ~15 GB of GPU memory)

server = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    cwd="/kaggle/working", env=os.environ.copy())
time.sleep(10)
tunnel = ngrok.connect(8501)
print("Open your app here:", tunnel.public_url)

2026-10-09 20:45:34.330 Port 8501 is not available


Open your app here: https://pristine-luminous-rinse.ngrok-free.dev


In [73]:
import os, subprocess, time
from pyngrok import ngrok

# stop the old app and tunnel
try:
    server.terminate(); server.wait(timeout=15)
except Exception as e:
    print("server already stopped:", e)
ngrok.kill()

# start again
os.environ["COACH_4BIT"] = "1"
server = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    cwd="/kaggle/working", env=os.environ.copy())
time.sleep(10)
tunnel = ngrok.connect(8501)
print("Open your app here:", tunnel.public_url)

2026-10-09 20:54:54.607 Uvicorn server started on :::8501



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.148.170.216:8501

Open your app here: https://pristine-luminous-rinse.ngrok-free.dev


Loading checkpoint shards: 100%|██████████| 3/3 [00:13<00:00,  4.62s/it]
